In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''

Created on 2024-07-12
Last modified on 2024-07-12
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener información a partir de los .md generados del bloque get_cti_information. Es requerido haber ejecutado previamente el módulo get_cti_information.

'''

'\n\nCreated on 2024-07-12\nLast modified on 2024-07-12\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener información a partir de los .md generados del bloque get_cti_information. Es requerido haber ejecutado previamente el módulo get_cti_information.\n\n'

**Requerimientos**

In [2]:
import os
import re

import pandas as pd

from stix2 import Filter, MemoryStore
import stix2
import requests

##### **Parámetros**

In [3]:
matrix = 'enterprise' # enterprise / ics / mobile
news_sources = ['thehackernews']# ['thehackernews','nist', 'cyble'] Lista de outputs a evaluar


##### **Funciones**

In [4]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    stix_json = requests.get(url).json()
    return MemoryStore(stix_data=stix_json["objects"])

In [5]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [6]:
def get_techniques_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id técnicas y nombre de técnicas.
    '''
    techniques_data = []
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    for tech in techniques:
        deprecated = False
        revoked = False
        description = ''
        if 'x_mitre_deprecated' in tech: 
            deprecated = tech['x_mitre_deprecated']
        if 'revoked' in tech: 
            revoked = tech['revoked']
        techniques_data.append({
            "technique_ID": tech['external_references'][0]['external_id'],
            "technique": tech['name'],
            "technique_deprecated": deprecated,
            "technique_revoked": revoked
        })
        
    techniques_df = pd.DataFrame(techniques_data)

    if revoked_deprecated:
        techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    
    techniques_df = techniques_df[['technique_ID','technique']]

    techniques_id_list = techniques_df['technique_ID'].tolist()
    techniques_name_list = techniques_df['technique'].tolist()

    return techniques_id_list, techniques_name_list

In [36]:
def get_tactics_lists(matrix_store):
    '''
    Función que retorna las listas de id táctica y nombre de táctica.
    '''
    tactics_data = []
    tactics = matrix_store.query([Filter('type', '=', 'x-mitre-tactic')])
    for tact in tactics:
        tactics_data.append({
            "tactic_ID": tact['external_references'][0]['external_id'],
            "tactic": tact['name']
        })

    tactics_df = pd.DataFrame(tactics_data)
    tactics_df = tactics_df[['tactic_ID','tactic']]

    tactics_id_list = tactics_df['tactic_ID'].tolist()
    tactics_name_list = tactics_df['tactic'].tolist()

    return tactics_id_list, tactics_name_list

In [43]:
def get_datasources_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id data source y nombre de data source.
    '''
    datasoruces_data = []
    data_sources = matrix_store.query([Filter('type', '=', 'x-mitre-data-source')])
    for ds in data_sources:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in ds: 
            deprecated = ds['x_mitre_deprecated']
        if 'revoked' in ds: 
            revoked = ds['revoked']

        datasoruces_data.append({
            "datasource_ID": ds['external_references'][0]['external_id'],
            "datasource": ds['name'],
            "datasource_deprecated": deprecated,
            "datasource_revoked": revoked
        })

    datasoruces_df = pd.DataFrame(datasoruces_data)
    if revoked_deprecated:
        datasoruces_df = datasoruces_df[(datasoruces_df['datasource_deprecated']!=True)&(datasoruces_df['datasource_revoked']!=True)]

    datasoruces_df = datasoruces_df[['datasource_ID','datasource']]
    datasoruces_id_list = datasoruces_df['datasource_ID'].tolist()
    datasoruces_name_list = datasoruces_df['datasource'].tolist()
    
    return datasoruces_id_list, datasoruces_name_list

In [71]:
def get_platforms_list(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las lista de plataformas.
    '''
    platforms_from_tech = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    platforms_data = []
    for technique in platforms_from_tech:
        platform_name = 'None'
        if 'x_mitre_data_sources' in technique: 
            try: 
                platform_name = technique['x_mitre_platforms']
            except:
                platform_name = 'None'

        platforms_data.append({
            "platform": platform_name,
        })
    # Generamos df a partir de los datos recopilados
    platforms_df = pd.DataFrame(platforms_data)
    platforms_df = platforms_df.explode('platform')
    platforms_df = platforms_df.drop_duplicates()
    platforms_df = platforms_df.sort_values(by='platform')
    platforms_df = platforms_df.reset_index(drop=True)
    platforms_name_list = platforms_df['platform'].tolist()
    
    return platforms_name_list

In [75]:
def get_software_lists(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna las listas de id software y nombre de software.
    '''
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    software_data = []
    for sw in software:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in sw: 
            deprecated = sw['x_mitre_deprecated']
        if 'revoked' in sw: 
            revoked = sw['revoked']

        software_data.append({
            "software_ID": sw['external_references'][0]['external_id'],
            "software": sw['name'],
            "software_deprecated": deprecated,
            "software_revoked": revoked
        })


    # Generamos df a partir de los datos recopilados
    software_df = pd.DataFrame(software_data)
    revoked_deprecated = True
    if revoked_deprecated:
        software_df = software_df[(software_df['software_deprecated']!=True)&(software_df['software_revoked']!=True)]
    
    software_df = software_df[['software_ID','software']]
    software_id_list = software_df['software_ID'].tolist()
    software_name_list = software_df['software'].tolist()
    
    return software_id_list, software_name_list

In [ ]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [55]:
def match_items(items_list, news_sources_list):
    '''
    Función encargada de devolver un diccionario compuesto por la ruta del archivo analizado como clave y como valor una lista que a su vez se compone por el numero de coincidencias encontradas y las propias coincidencias (únicas).
    '''
    items_list = [item.lower() for item in items_list]
    results = {}
    for ns in news_sources_list: # recorremos la lista de outputs (thehackernews, nist, cyble)
        review_path = os.path.join(os.path.dirname(os.getcwd()), 'get_cti_information', 'outputs', ns)
        items_md = get_list_of_files_sub(review_path)
        items_md = [item for item in items_md if os.path.splitext(item)[1] == '.md']# eliminamos posibles archivos cuya extension no sea .md 
        
        for md_file in items_md:
            try:
                # Comenzamos intentando abrir el fichero, en caso de que no se pueda generamos excepción y se revisa el nombre del fichero antes de copiar el fichero a la carpeta T0000
                with open(md_file, 'r', encoding='utf-8') as file:
                    md_content = file.read()
                    md_content = md_content.lower()
                    md_content = re.sub(r'---.*?---', '', md_content, flags=re.DOTALL) # Eliminamos el header del contenido que vamos a revisar
                    items_found = find_content_in_list(md_content, items_list)
                    results[md_file] = [len(items_found), items_found]
            except:
                pass
    return results

In [54]:
def find_content_in_list(texto, items_list):
    '''
    Función encargada de buscar en un texto facilitado los elementos contenidos en la lista de técnicas. Retorna una lista con el contenido encontrado.
    '''
    if isinstance(texto, str):
        found = set()  # Usar un conjunto para almacenar elementos únicos
        for item in items_list:
            if item in texto:
                found.add(item.upper())  # Añadir el elemento al conjunto
        return list(found)  # Convertir el conjunto a lista antes de devolver
    else:
        return []

In [33]:
def results_to_df(dict_results):
    '''
    Función para convertir en df el diccionario generado por match_items()
    '''
    data_tuples = [(key, *value) for key, value in dict_results.items()]
    dict_results_df = pd.DataFrame(data_tuples, columns=['file_path', 'match_count', 'matches'])
    dict_results_df = dict_results_df.sort_values('match_count', ascending=False)
    return dict_results_df

### **Ejecución principal**

In [7]:
mitre_matrix = get_data_from_branch(matrix)

In [ ]:
techniques_id_list, techniques_name_list =  get_techniques_lists(mitre_matrix)

In [40]:
tactics_id_list, tactics_name_list =  get_tactics_lists(mitre_matrix)

In [68]:
datasources_id_list, datasources_name_list =  get_datasources_lists(mitre_matrix)

In [72]:
platforms_name_list =  get_platforms_list(mitre_matrix)

In [79]:
software_id_list, software_name_list =  get_software_lists(mitre_matrix)

#### **Búsqueda de técnicas (ID y nombre)**

In [59]:
results_techniques_id = match_items(techniques_id_list, news_sources)
results_to_df(results_techniques_id).head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


In [57]:
results_techniques_name = match_items(techniques_name_list, news_sources)
results_to_df(results_techniques_name).head(3)

,file_path,match_count,matches
10,c:\Users\jelopez\Documents\CyberProof\python\d...,8,"[BRUTE FORCE, VULNERABILITIES, AT, CREDENTIALS..."
11,c:\Users\jelopez\Documents\CyberProof\python\d...,8,"[BRUTE FORCE, VULNERABILITIES, AT, CREDENTIALS..."
3,c:\Users\jelopez\Documents\CyberProof\python\d...,7,"[MALWARE, SERVER, AT, CREDENTIALS, SCHEDULED T..."


#### **Búsqueda de táctica (ID y nombre)**

In [63]:
results_tactics_id = match_items(tactics_id_list, news_sources)
results_to_df(results_tactics_id).head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


In [62]:
results_tactics_name = match_items(tactics_name_list, news_sources)
results_to_df(results_tactics_name).head(3)

,file_path,match_count,matches
7,c:\Users\jelopez\Documents\CyberProof\python\d...,5,"[PERSISTENCE, IMPACT, EXECUTION, DISCOVERY, LA..."
4,c:\Users\jelopez\Documents\CyberProof\python\d...,4,"[IMPACT, EXECUTION, DISCOVERY, LATERAL MOVEMENT]"
5,c:\Users\jelopez\Documents\CyberProof\python\d...,4,"[IMPACT, EXECUTION, DISCOVERY, LATERAL MOVEMENT]"


#### **Búsqueda de data sources (ID y nombre)**

In [61]:
results_datasources_id = match_items(datasources_id_list, news_sources)
results_to_df(results_datasources_id).head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


In [60]:
results_datasources_name = match_items(datasources_name_list, news_sources)
results_to_df(results_datasources_name).head(3)

,file_path,match_count,matches
2,c:\Users\jelopez\Documents\CyberProof\python\d...,5,"[COMMAND, SERVICE, PROCESS, FILE, SCRIPT]"
9,c:\Users\jelopez\Documents\CyberProof\python\d...,5,"[SERVICE, FILE, GROUP, SCRIPT, PERSONA]"
0,c:\Users\jelopez\Documents\CyberProof\python\d...,4,"[GROUP, PERSONA, SERVICE, FILE]"


#### **Búsqueda de plataformas (sólo nombre)**

In [73]:
results_platforms_name = match_items(platforms_name_list, news_sources)
results_to_df(results_platforms_name).head(3)

,file_path,match_count,matches
5,c:\Users\jelopez\Documents\CyberProof\python\d...,3,"[LINUX, PRE, NETWORK]"
0,c:\Users\jelopez\Documents\CyberProof\python\d...,2,"[PRE, WINDOWS]"
3,c:\Users\jelopez\Documents\CyberProof\python\d...,2,"[PRE, WINDOWS]"


#### **Búsqueda de software (ID y nombre)**

In [81]:
results_software_id = match_items(software_id_list, news_sources)
results_to_df(results_software_id).head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
1,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]
2,c:\Users\jelopez\Documents\CyberProof\python\d...,0,[]


In [82]:
results_software_name = match_items(software_name_list, news_sources)
results_to_df(results_software_name).head(3)

,file_path,match_count,matches
0,c:\Users\jelopez\Documents\CyberProof\python\d...,7,"[CONTI, DISCO, EXPAND, AT, NET, REG, TOR]"
1,c:\Users\jelopez\Documents\CyberProof\python\d...,7,"[CONTI, DISCO, AT, TOR, NET, REG, ROUTE]"
2,c:\Users\jelopez\Documents\CyberProof\python\d...,7,"[CONTI, PING, DISCO, AT, NET, REG, TOR]"
